# 工具契约与错误：结构、权限和执行结果分开看

使用[共享Schema](../05-code/shared-schemas/README.md)与[TS Runtime](../05-code/tool-runtime-typescript/README.md)。Python需要`jsonschema`；两个TS工程提前`npm ci`。本实验无模型调用、无外部搜索。

In [1]:
from pathlib import Path
import sys, json, shutil
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "10-Knowledge").is_dir())
from jsonschema import Draft202012Validator
SCHEMAS = ROOT / "10-Knowledge/05-tools-skills-protocols/05-code/shared-schemas"
NPM = shutil.which("npm") or shutil.which("npm.cmd")
NODE = shutil.which("node")
assert NPM and NODE, "Node.js and npm must be available on PATH"


## 先执行同一组跨语言正反例

每个Schema有一个正确对象与一个带额外字段的错误对象。AJV测试随后使用同样文件，避免Python和TS各自维护不同预期。

In [2]:
checks = []
for path in sorted(SCHEMAS.glob("*.schema.json")):
    name = path.name.removesuffix(".schema.json")
    validator = Draft202012Validator(json.loads(path.read_text(encoding="utf-8")))
    for kind in ["valid", "invalid"]:
        sample = json.loads((SCHEMAS / "examples" / f"{name}.{kind}.json").read_text(encoding="utf-8"))
        accepted = validator.is_valid(sample)
        checks.append((name, kind, accepted))
        assert accepted == (kind == "valid")
print(*checks, sep="\n")
print("校验对象总数:", len(checks))

('agent-state', 'valid', True)
('agent-state', 'invalid', False)
('eval-task', 'valid', True)
('eval-task', 'invalid', False)
('tool-call', 'valid', True)
('tool-call', 'invalid', False)
('tool-result', 'valid', True)
('tool-result', 'invalid', False)
('trace-event', 'valid', True)
('trace-event', 'invalid', False)
('trial-result', 'valid', True)
('trial-result', 'invalid', False)
校验对象总数: 12


## 合法JSON不等于合法工具参数

空格可通过字符串长度校验，但应由搜索业务逻辑拒绝；它说明Schema有明确边界。

In [3]:
schema = {"type":"object", "properties":{"query":{"type":"string","minLength":1}}, "required":["query"], "additionalProperties":False}
validator = Draft202012Validator(schema)
for args in [{"query":"上下文"}, {"query":7}, {"query":" "}, {"query":"x", "admin":True}]:
    structure_ok = validator.is_valid(args)
    business_ok = structure_ok and bool(args["query"].strip())
    print(args, "结构:", structure_ok, "业务:", business_ok)

{'query': '上下文'} 结构: True 业务: True
{'query': 7} 结构: False 业务: False
{'query': ' '} 结构: True 业务: False
{'query': 'x', 'admin': True} 结构: False 业务: False


## 编译并执行真实TS Runtime测试

下方启动子进程执行npm test，返回码非0则实验失败。包含取消与超时、权限、并发去重和Schema兼容，不把手写预期作为测试结果。

In [4]:
import subprocess
BASE = ROOT / "10-Knowledge/05-tools-skills-protocols/05-code"
for project in ["tool-runtime-typescript", "mcp-server-typescript"]:
    result = subprocess.run([NPM, "test"], cwd=BASE / project, text=True, encoding="utf-8", capture_output=True, timeout=120)
    print(project, "exit:", result.returncode)
    print(result.stdout)
    assert result.returncode == 0, result.stderr

tool-runtime-typescript exit: 0

> tool-runtime-typescript@0.1.0 test
> npm run build && node --test dist/test/*.test.js


> tool-runtime-typescript@0.1.0 build
> tsc

(node:8832) Warning: The 'NO_COLOR' env is ignored due to the 'FORCE_COLOR' env being set.
(Use `node --trace-warnings ...` to show where the warning was created)
✔ input/output contracts and permissions (37.6781ms)
✔ concurrent duplicates execute once and conflicts are rejected (13.619ms)
✔ timeout aborts cooperative handler and is not automatically retryable (31.9557ms)
✔ caller cancellation propagates (31.3969ms)
✔ unknown tool and exceptions stay structured (30.213ms)
✔ caller mutation cannot change validated input or poison cached output (12.4818ms)
✔ cancellation before handler scheduling prevents execution (12.1335ms)
✔ the full ToolCall envelope is checked before any handler executes (15.2293ms)
✔ malformed host identity is rejected without executing or throwing (12.1826ms)
(node:6860) Warning: The 'NO_COLOR' env

mcp-server-typescript exit: 0

> mcp-server-typescript@0.1.0 test
> npm run build && node --test dist/test/*.test.js


> mcp-server-typescript@0.1.0 build
> tsc

(node:8644) Warning: The 'NO_COLOR' env is ignored due to the 'FORCE_COLOR' env being set.
(Use `node --trace-warnings ...` to show where the warning was created)
✔ SDK discovery, arguments, structured data and execution error (53.3469ms)
✔ real stdio subprocess starts, lists, calls and closes (672.1169ms)
ℹ tests 2
ℹ suites 0
ℹ pass 2
ℹ fail 0
ℹ cancelled 0
ℹ skipped 0
ℹ todo 0
ℹ duration_ms 1086.2504



## 真实MCP stdio调用

固定TypeScript SDK 2.0.0。真实stdio客户端使用自动版本协商并断言2026-07-28；内存测试另以legacy模式覆盖2025-11-25兼容路线。见[MCP版本说明](../01-concepts/03-mcp.md)。

In [5]:
result = subprocess.run([NODE, "dist/src/client.js", "上下文"], cwd=BASE / "mcp-server-typescript", text=True, encoding="utf-8", capture_output=True, timeout=30)
assert result.returncode == 0, result.stderr
response = json.loads(result.stdout)
print(json.dumps(response, ensure_ascii=False, indent=2))
assert response["protocolVersion"] == "2026-07-28"
assert response["result"]["structuredContent"]["documents"][0]["id"] == "doc-1"

# 工具正常执行但没有命中，与业务拒绝是两种不同结果。
for query in ["不存在的教学关键词", " "]:
    completed = subprocess.run([NODE, "dist/src/client.js", query], cwd=BASE / "mcp-server-typescript", text=True, encoding="utf-8", capture_output=True, timeout=30)
    assert completed.returncode == 0, completed.stderr
    payload = json.loads(completed.stdout)["result"]
    print(repr(query), json.dumps(payload, ensure_ascii=False))
    if query.strip():
        assert not payload.get("isError", False)
        assert payload["structuredContent"]["documents"] == []
    else:
        assert payload["isError"] is True


{
  "protocolVersion": "2026-07-28",
  "tools": [
    "search_docs"
  ],
  "result": {
    "_meta": {
      "io.modelcontextprotocol/serverInfo": {
        "name": "learning-docs",
        "version": "0.1.0"
      }
    },
    "content": [
      {
        "type": "text",
        "text": "{\"documents\":[{\"id\":\"doc-1\",\"text\":\"上下文预算为输出和工具结果预留空间。\"}]}"
      }
    ],
    "structuredContent": {
      "documents": [
        {
          "id": "doc-1",
          "text": "上下文预算为输出和工具结果预留空间。"
        }
      ]
    }
  }
}


'不存在的教学关键词' {"_meta": {"io.modelcontextprotocol/serverInfo": {"name": "learning-docs", "version": "0.1.0"}}, "content": [{"type": "text", "text": "{\"documents\":[]}"}], "structuredContent": {"documents": []}}


' ' {"_meta": {"io.modelcontextprotocol/serverInfo": {"name": "learning-docs", "version": "0.1.0"}}, "content": [{"type": "text", "text": "query must contain non-whitespace characters"}], "isError": true}


## 如何使用这些结果

参数错误应修参数，权限不足应走真实授权，超时应先核查后端是否执行。测试证明这些已构造场景在本实现中可处理；HTTP部署、真实身份服务与跨进程幂等仍需单独验证。